# 10 - Matriz completa de experimentos
En este notebook ejecutamos la matriz completa de experimentos: todos los conjuntos de variables (`config.FEATURE_SETS`, 4) × todos los perfiles (`config.PROFILES`, 4) × los modelos de `config.MATRIX_MODELS` (5), en las 17 particiones. Son 17 × 4 × 4 × 5 = **1.360 filas**, una por modelo y pliegue.

Permite comparar los subconjuntos con el conjutno completo, por lo que sirve de base para contrastar las hipótesis de investigación.

## Carga de datos y conjuntos de variables

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from urban_accessibility_ml.config import FEATURE_SETS, MATRIX_MODELS, PROCESSED_DATA_DIR, PROFILES, REPORTS_DIR
from urban_accessibility_ml.modeling.evaluate import summarize_folds
from urban_accessibility_ml.modeling.experiments import run_experiments, save_results
from urban_accessibility_ml.plotting.palettes import SERIE_BASE
from urban_accessibility_ml.plotting.plots import SERIES, save_figure, use_style
from urban_accessibility_ml.preprocessing.features import dimensions_for
from urban_accessibility_ml.preprocessing.split import build_all_splits

RESULTS_FILE = REPORTS_DIR / "results" / "0.10-experiment-matrix.csv"
STRATEGIES = ["within_city", "mixed", "cross_city"]
# Refutation criterion of the thesis: "clearly inferior" = below 80 % of the F1 macro of the complete set
CLEARLY_INFERIOR_RATIO = 0.8
FEATURE_SET_COLORS = dict(zip(FEATURE_SETS, SERIES))

use_style()

df = pd.read_csv(PROCESSED_DATA_DIR / "streets_unified.csv")
pd.Series({feature_set: dimensions_for(feature_set) for feature_set in FEATURE_SETS}, name="dimensiones")

Com nota, con el umbral actual (`config.THRESHOLD`, que es 50%), el conjunto `combinado` tiene las mismas dimensiones que `calle`, así que sus resultados serán idénticos y la hipótesis H3 no se puede contrastar.

## Particiones

In [ ]:
splits = build_all_splits(df)
len(splits)

## Entrenamiento y evaluación
Ejecutamos `run_experiments` con los valores por defecto.

In [ ]:
results = run_experiments(df, splits)
save_results(results, RESULTS_FILE)
print(len(results), "filas =", len(splits), "particiones x", len(FEATURE_SETS), "conjuntos x", len(PROFILES), "perfiles x", len(MATRIX_MODELS), "modelos")
results.head()

## Resumen por conjunto, perfil y modelo
Calculamos la media y desviación típica entre pliegues, separadas por estrategia (`within_city`, `cross_city` y `mixed`).

In [ ]:
summary = summarize_folds(results, ["strategy", "feature_set", "profile", "model"])
summary.round(3)

## Matriz agregada de F1 macro medio

Vamos a utilizar la F1 macro como comparador en una matriz agregada.

In [ ]:
summary.pivot(index=["strategy", "feature_set", "profile"], columns="model", values="f1_macro_mean").round(3)

## F1 macro por cada conjunto de variables
Vamos a mostrar en un gráfico el mejor modelo de cada conjunto de variables (el de mayor F1 macro medio), para cada estrategia y perfil, mostrando la desviación típica entre pliegues con una barra vertical negra.

La línea discontinua marca el 80 % de la F1 macro del conjunto `completo`: por lo que si la barra de uno de los subconjuntos queda por debajo de esa línea discontinua, se podrían descartar algunas de las hipótesis de la memoria.

In [ ]:
# Best model of each strategy, feature set and profile (by mean F1 macro)
best = summary.loc[summary.groupby(["strategy", "feature_set", "profile"])["f1_macro_mean"].idxmax()]

x = np.arange(len(PROFILES))
width = 0.2

fig, axes = plt.subplots(1, len(STRATEGIES), figsize=(13, 4.6), sharey=True)
for ax, strategy in zip(axes, STRATEGIES):
    data = best[best["strategy"] == strategy]
    for i, feature_set in enumerate(FEATURE_SETS):
        rows = data[data["feature_set"] == feature_set].set_index("profile").loc[PROFILES]
        ax.bar(
            x + (i - (len(FEATURE_SETS) - 1) / 2) * width, rows["f1_macro_mean"], width,
            yerr=rows["f1_macro_std"].fillna(0), color=FEATURE_SET_COLORS[feature_set], label=feature_set,
            error_kw={"linewidth": 1},
        )
    complete = data[data["feature_set"] == "completo"].set_index("profile").loc[PROFILES, "f1_macro_mean"]
    ax.hlines(
        CLEARLY_INFERIOR_RATIO * complete, x - 0.4, x + 0.4,
        colors="black", linestyles="--", linewidth=1, label="80 % de completo",
    )
    ax.set_title(strategy)
    ax.set_xticks(x, PROFILES)
    ax.set_ylim(0, 1)
axes[0].set_ylabel("F1 macro (mejor modelo)")
handles, names = axes[0].get_legend_handles_labels()
fig.legend(handles, names, loc="lower center", ncol=len(names), frameon=False)
fig.tight_layout(rect=(0, 0.06, 1, 1))
save_figure("f1_macro_por_conjunto", "0.10-matriz-experimentos", fig=fig)

Las barras verticales negras indican la desviación típica.

Vemos que la estrategia de transferencia entre ciudades (`cross_city`) es la que obtiene peores resultados en términos absolutos, con F1 macro inferiores a los de las otras 2 estrategias. Lo que parece indicar que la hipótesis de transferencia entre ciudades tiende a refutarse.

Las otras dos estrategias parecen más optimistas, a excepción de la estrategia mixta para el conjunto de dimensiones cartográfico y aéreo, en cognitiva y global.

## Comprobación: `combinado` frente a `calle`

Vamos a comprobar a partir de los datos que con el umbral utilizado, el conjunto combinado es igual al que es obtenible a partir de las imágenes a pie de calle.

In [ ]:
wide = summary.pivot(index=["strategy", "profile", "model"], columns="feature_set", values="f1_macro_mean")
print("combinado == calle en todos los casos:", bool((wide["combinado"] == wide["calle"]).all()))